[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tiejun-ai/ai_websearch/blob/main/v1/ai_websearch.ipynb)

# AI Web Search (v1)

A minimal RAG-based AI web search demo.

**Workflow:**
1. User submits a query
2. **Web Search** — Tavily fetches the top 10 results (title, URL, snippet)
3. **LLM Generation** — a single LiteLLM call produces:
   - **Answer**: synthesized response with inline citations
   - **Web Search Results**: all 10 results listed with clickable titles
4. **Display** — markdown output converted to HTML and rendered in Colab

In [ ]:
# Install required packages
!pip install tavily-python litellm markdown --quiet

## Configuration

Set your API keys and choose a model before running.

- **TAVILY_API_KEY**: get one at [tavily.com](https://tavily.com)
- **OPENAI_API_KEY**: get one at [platform.openai.com](https://platform.openai.com)
- **MODEL**: any LiteLLM-supported model string (default: `gpt-4o-mini`, cheap and capable)

In [ ]:
import os

TAVILY_API_KEY = "tvly-..."   # your Tavily API key
OPENAI_API_KEY = "sk-..."     # your OpenAI API key

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY  # make key available to LiteLLM

MODEL = "gpt-4o-mini"  # change to e.g. "gpt-4o", "claude-3-haiku-20240307"
TOP_K = 10             # number of web search results to retrieve

## Step 1: Web Search

The `search_web` function calls the Tavily API and returns the top `k` results.
Each result contains a **title**, **URL**, and a **content snippet**.

In [ ]:
from tavily import TavilyClient

def search_web(query, k=TOP_K):
    """Search the web and return top-k results from Tavily."""
    client = TavilyClient(api_key=TAVILY_API_KEY)
    response = client.search(query, max_results=k)
    # Each result dict: {"title": str, "url": str, "content": str}
    return response["results"]

## Step 2: Generate Answer with LLM

The `generate_answer` function builds a prompt with all search results and asks the LLM to:
- Answer the query using only the provided results (skip irrelevant ones)
- Add **inline citations** as markdown links `[Title](URL)` near relevant text
- End with a **Web Search Results** section listing every result

In [ ]:
import litellm
litellm.set_verbose = False  # suppress debug output

def generate_answer(query, results, model=MODEL):
    """Send query + search results to LLM; return a markdown-formatted answer."""
    # Format each result for the prompt, numbered 1..n
    results_text = "\n\n".join(
        f"[{i+1}] Title: {r['title']}\nURL: {r['url']}\nContent: {r['content']}"
        for i, r in enumerate(results)
    )
    n = len(results)

    system_prompt = (
        "You are a helpful research assistant. "
        "Answer the user's query faithfully using ONLY the provided search results. "
        "Do not consider results that are irrelevant to the query. "
        "Add inline citations using the result number as the anchor text: [n](URL) "
        "(e.g. [1](https://example.com)) near supporting text. "
        f"End with a '## Web Search Results' section listing ALL {n} results as:\n"
        "### n. [Title](URL)\nSnippet text"
    )
    user_prompt = f"Question: {query}\n\nSearch Results:\n{results_text}"

    response = litellm.completion(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
    )
    return response.choices[0].message.content  # markdown string

## Step 3: Display the Answer

Convert the markdown answer to HTML and render it inline in the notebook.
This makes citations clickable and the answer easy to read inside Colab.

In [ ]:
import markdown
from IPython.display import display, HTML

def display_answer(answer_md):
    """Convert markdown answer to HTML and render it in the notebook."""
    html = markdown.markdown(answer_md, extensions=["extra"])
    display(HTML(html))

## Run the Search

Edit the `query` below and run this cell. The answer will be rendered as HTML.

In [ ]:
query = "What are the latest AI breakthroughs in 2025?"  # <- change this

# Step 1: fetch top-10 web results
results = search_web(query)

# Step 2: generate answer with inline citations
answer_md = generate_answer(query, results)

display_answer(answer_md)  # rendered HTML in Colab